### ⚠️ Author's Note 
Please refer to `README.md` file for necessary installations and setup

In [ ]:
!pip install numpy pandas matplotlib opencv-python seaborn tensorflow tensorboard python-dotenv scikit-learn -q

In [ ]:
# Run to utilize GPU for TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

In [ ]:
# General Libraries
import numpy as np
import pandas as pd
# import itertools
import random
from pathlib import Path
# import os.path

# Visualization Libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
import seaborn as sns

# Tensorflow Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image 
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

sns.set_style('darkgrid')


In [ ]:
import os

# Reproducibility for future cases
def set_seed(seed=123):
    # Standardize seed values
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Limits to single thread for better results
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)

    # Forces deterministic operations
    os.environ['PYTHONHASHSEED'] = str(seed)
    tf.config.experimental.enable_op_determinism()

set_seed(seed=123)

### ⚠️ Author's Note 

With a <font color="#6A89A7">SSLCertVerificationError</font> on macOS with `python.org` Python, either run:   
1.`Install Certificates.command` in the Python's Applications folder   
2. Run the following code in the terminal, replacing `3.X` with the installed version such as `3.13`
```bash
/Applications/Python\ 3.X/Install\ Certificates.command 
```

In [ ]:
import urllib.request
url = 'https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py'
filename = 'helper_functions.py'
urllib.request.urlretrieve(url, filename)

# Import helper functions for loading and visualizing data
from helper_functions import create_tensorboard_callback, plot_loss_curves, unzip_data, compare_historys, walk_through_dir, pred_and_plot

### Using `kagglehub` to download the dataset
1. Generate API key named `kaggle.json` from the Kaggle account settings (*Profile* $\rightarrow$ *Account* $\rightarrow$ *Create New API Token*)
2. Add Kaggle credentials to `.env`

```js
KAGGLE_USERNAME=your_kaggle_username
KAGGLE_KEY=your_api_token
```

3. `pip install kagglehub` library in the terminal (if haven't yet)
* Downloads Kaggle datasets
* Uses the Kaggle credentials from `.env`
```

In [ ]:
pip install kagglehub

In [ ]:
import kagglehub
from dotenv import load_dotenv

# Load dotenv
load_dotenv()

# Download required asset
dataset = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")

# Verify the contents of the dataset path
def walk_through_dir(dir_path):

    base_dir = os.path.dirname(dir_path)
    
    for dirpath, dirnames, filenames in os.walk(dir_path):
        relative_path = os.path.relpath(dirpath, base_dir)
        
        print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{relative_path}'.")

# !ls -R {dataset}
walk_through_dir(dataset)

## Data Source: Healthy and Bleached Corals Dataset 🪸
This medium dataset includes `923` total collected images used with `Flickr API`. The images have a consistent max size of 300 pixels and are categorized into two distict classes:
* Healthy corals (`438` images)
* Bleached corals (`485` images)  



### ⚠️ Author's Note 
Many of the images are close-ups with subtle class distinction due to the naturally white or earth-toned corals.  While the dataset uses lower color saturation and visibility, it simulates the real-world marine condition.

This image dataset is evaluated and corrected, eliminating near duplicate images to reduce over-estimation of the model's accuracy.

Hence, accurate labeling task is more difficult to achieve.

#### Example:  
* bleached_corals/10160888014_be7c71167c_o.jpg  

* healthy_corals/10160888014_be7c71167c_o.jpg

## 📊 Constructing a DataFrame  
|       | Filepaths of Images      | Class Labels of Images |
| :---  | :---:                    |       :---:            |
| 0     | /healthy-and-bleached... | healthy_corals         |
| ...   |           ...            |       ...              |
| 918   | /healthy-and-bleached... | bleached corals        |  


In [ ]:
from pathlib import Path

def convert_path_to_df(dataset):
    image_dir = Path(dataset)

    # Get filepaths and direct class labels
    extensions = ['.jpg', '.jpeg', '.png']
    filepaths = [str(i) for i in image_dir.rglob('*') if i.suffix.lower() in extensions]

    labels = [Path(p).parent.name.replace('_', ' ') for p in filepaths]

    # Construct DataFrame
    return pd.DataFrame({'Filepath': filepaths,'Label': labels})

image_df = convert_path_to_df(dataset)
print("DataFrame constructed.")

In [ ]:
# Scan for corrupted images
import PIL
from pathlib import Path
from PIL import UnidentifiedImageError

extensions = [".jpg", ".jpeg", ".png"]
corrupt_count = 0

for img_path in Path(dataset).rglob("*"):
  if img_path.suffix.lower() in extensions:
    try:
        (PIL.Image.open(img_path)).verify
    except (PIL.UnidentifiedImageError, OSError):
            print(f"Corrupt file found: {img_path}")
            os.remove(img_path)
            corrupt_count += 1
print(f"Scan complete. Found {corrupt_count} corrupt file(s).")

## ⚖️ Error Level Analysis (ELA)

1. Compress an image as a <font color="#D08C35">JPEG</font> at a given quality parameter  
2. Compute the  <font color="#D08C35">absolute pixel difference</font> between the original and compressed images  
3.  <font color="#D08C35">Multiply</font> the difference by a  <font color="#D08C35">fixed scale factor</font>
4. Return an  <font color="#D08C35">ELA image  </font>

In [ ]:
def compute_ela_cv(path, quality):
    temp_file = 'temp_file.jpeg'
    SCALE = 15

    # Keep native BGR for loading and preventing channel swapping
    orig_img = cv2.imread(path)
    cv2.imwrite(temp_file, orig_img, [cv2.IMWRITE_JPEG_QUALITY, quality])

    compressed_img = cv2.imread(temp_file)

    # Multiply scale to absolute difference between images
    diff = SCALE * cv2.absdiff(orig_img, compressed_img)

    # Convert final output to RGB for standard saving
    return cv2.cvtColor(diff, cv2.COLOR_BGR2RGB)

## 🖥️ Visual Tradeoff  
<font color="#788BFF"> Digital images degrade uniformly for each JPEG save.</font>  
A new spliced object will have a different compression history than its background.  <br></br>
Saving the image multiple times:
* Shows a mathematical difference over time
* Areas with uniform brightness likely belongs to the original image
* Specific section with bright flashes or differing appearance than the rest of the image suggests external insertion (e.g. not part of original image)  

With `SCALE = 15`:
* An image with very low compression errors yields a darker ELA image
* Highly accurate for cross-image comparisons  
  
A multi-panel layout compares the visual differences between an original image and its recompressed output 
* Displays random image from healthy and bleached corals

In [ ]:
# Load and normalize random healthy coral
random_row = image_df.query("Label == 'healthy corals'").sample(n=1).iloc[0]
image_path = Path(random_row['Filepath'])
label = random_row['Label']
orig = cv2.imread(image_path)
orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB) / 255.0

columns = 3
rows = 2
fig = plt.figure(figsize=(15, 10))

# Display original image
ax = fig.add_subplot(rows, columns, 1)
ax.title.set_text(f'Original Image of a Healthy Coral')
plt.imshow(orig)
plt.axis('off')

init_val = 100
for i in range(2, columns * rows + 1):
    quality = init_val - (i - 2) * 8
    img = compute_ela_cv(path=image_path, quality=quality) / 255.0

    ax = fig.add_subplot(rows, columns, i)
    ax.title.set_text(f'ELA Quality: {quality}')
    plt.imshow(img)
    plt.axis('off')

print("Random Image Path (Healthy Corals): " + str(image_path))
plt.tight_layout()
plt.show()

In [ ]:
# Load and normalize random bleached coral
random_row = image_df.query("Label == 'bleached corals'").sample(n=1).iloc[0]
image_path = Path(random_row['Filepath'])
label = random_row['Label']

orig = cv2.imread(image_path)
orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB) / 255.0

columns = 3
rows = 2
fig = plt.figure(figsize=(15, 10))

# Display original image
ax = fig.add_subplot(rows, columns, 1)
ax.title.set_text(f'Original Image of a Bleached Coral')
plt.imshow(orig)
plt.axis('off')

# Iterate through decreasing ELA Quality
init_val = 100
for i in range(2, columns * rows + 1):
    quality = init_val - (i - 2) * 8
    img = compute_ela_cv(path=image_path, quality=quality) / 255.0

    ax = fig.add_subplot(rows, columns, i)
    ax.title.set_text(f'ELA Quality: {quality}')
    plt.imshow(img)
    plt.axis('off')

print("Random Image Path (Bleached Corals): " + str(image_path))
plt.tight_layout()
plt.show()

## 📚 Splitting Data
**Split the data into three different categories:** 
1. Training data: Utilized for deep learning in a CNN model 
2. Validation data: Utilized for fine-tuning the parameters (i.e. weights biases) 
3. Testing data: Unseen data by the model that evaluates its performance    
  
This gives a 3-way split (e.g ~64% train/ ~16% val/20% test)
  
* Shuffle: Mixes rows randomly to break up any pre-existing order/patterns  
* Stratify: Splits shuffled rows into training and testing bins while strictly enforcing category ratios

#### Model should not learn accidental patterns based on data order


In [ ]:
# Separate training, validation, and testing data    
train_df, test_df = train_test_split(
    image_df, test_size=0.2, stratify=image_df['Label'], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df['Label'], random_state=42
)

## 🛠️ Preprocessing Images for VGG-19
Input images are <font color="#788BFF">converted from RBG to BGR and zero-centered</font> within each color channel without scaling.  
   
Zero-centering <font color="#788BFF">subtracts the specific mean pixel value of each channel (RGB)</font> calculated across the training set from every image.   
* <font color="#788BFF">Shifts pixel value distribution</font> so the mean of each channel becomes zero
* <font color="#788BFF">Removes bias</font> by stopping the model from seeing high or low values as a default state  
* Increases speed and accuracy of computational learning  
* Matches standard models


In [ ]:
BATCH_SIZE = 32
TARGET_SIZE = (224, 224)

In [ ]:
class_names = sorted(train_df['Label'].unique())

# Convert labels into integer IDs
label_encoder = tf.keras.layers.StringLookup(
    vocabulary=class_names,
    output_mode='int'
)

# Data augmentation 
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Resize and normalize
def load_and_preprocess(path, label, training=False):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, TARGET_SIZE)
    if training:
        image = augment(image, training=True)
    image = tf.keras.applications.vgg19.preprocess_input(image)
    return image, label_encoder(label) - 1


def make_dataset(df, shuffle=False, training=False):
    paths = df['Filepath'].to_numpy()
    labels = df['Label'].to_numpy()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)
    ds = ds.map(lambda x, y: load_and_preprocess(x, y, training=training), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

In [ ]:
# Check each dataset split
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:

    # Filter for existing files
    existing_files = [p for p in df['Filepath'] if os.path.exists(p)]
    missing_count = len(df) - len(existing_files)

    # Print splits of validated images and missing files
    print(f"{name.capitalize()} Split:")
    print(f"  - Found {len(existing_files)} validated image filenames belonging to {df['Label'].nunique()} classes.")

    if missing_count > 0:
        print(f"  - WARNING: {missing_count} files are missing from disk!!")
    print(" ")


## 🎯 Training the VGG-19 Model
TensorBoard Callbacks log training metrics (loss & accuracy) and weights during model training.
* Model Checkpoint: <font color="#CD7F6D">Saves model state</font> at a specific time, usually for best results and capturing optimal weights
* Early Stopping: <font color="#CD7F6D">Halts training</font> when `val_loss` does not improve after 3 epochs
* Reduce LR on Plateau: Adapts in real-time to training progress by <font color="#CD7F6D">lowering learning rate</font> as validation slows down
  
Hyperparameter Used:
* **Batch size:** 32
* **Epochs:** 100
* **Input Shape:** (224, 224, 3)
* **Output layer:** 2

In [ ]:
# Load the pretained model
pretrained_model = tf.keras.applications.vgg19.VGG19(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='max'
)

pretrained_model.trainable = False

#### Model Checkpoint Notes:
* `.weights` holds model parameters but not the complete model configuration
* `.h5` indicates Hierarchical Data Format ver.5 (HDF5), the traditional standard for Keras weights
* Saving the weights creates a smaller file size and optimizes storage efficiency  

In [ ]:
# Model Checkpoint
checkpoint = ModelCheckpoint(
  'checkpoint.weights.h5',
  save_weights_only=True,
  monitor="val_accuracy",
  save_best_only=True,
  verbose=1
)

# Early Stopping
early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    restore_best_weights = True
)

# Reduce LR on Plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2, 
    patience=3, 
    min_lr=1e-10
)

In [ ]:
model_inputs = Input(shape=(TARGET_SIZE[0], TARGET_SIZE[1], 3), name="model_input")

# Augment layers directly
x = layers.RandomFlip("horizontal")(model_inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomContrast(0.1)(x)

# Pass to pretrained model for feature extraction
x = pretrained_model(x, training=False)

# Classification layers
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.45)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.45)(x)

outputs = Dense(2, activation='softmax')(x)


model = Model(inputs=model_inputs, outputs=outputs)
model.compile(
    optimizer=Adam(0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Pass callbacks to model training
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[
        early_stopping,
        create_tensorboard_callback("training_logs",
                                    "corals_classification"),
        checkpoint,
        reduce_lr
    ]
)

## Model Evaluation
Use test data set to evaluate the performance of the model
* $TP$ refers to true positives (correct predictions)
* $FP$ refers to false positives
* $TP+FP$ refers to the total amount of relevant results
* $FN$ refers to false negatives
  
Different Classifications:
* Accuracy: Measures fractions of correct predictions
* Precision: Averaged among the classes $P=\frac{TP}{TP+FP}$
* Recall: Averaged among the classes $R=\frac{TP}{TP+FN}$
* F1 score: Harmonic mean of precision and recall; Averaged among the classes
$F1=2\times\frac{TP\times FP}{TP+FP}$

In [ ]:
results = model.evaluate(test_ds, verbose=0)

print("    Test Loss: {:.5f}".format(results[0]))
print("Test Accuracy: {:.2f}%".format(results[1] * 100))

Notice: Signs of overfitting 
* Add dropout to randomly deactivate neurons to learn robust features.
* Apply weight decay (i.e. L2 regularization with small and simple weights) 
* Augment data to increase the diversity of training set for generalization

## 📉 Visualizing Loss Curves

In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(accuracy))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epochs, accuracy, 'b', label='Training accuracy')
ax1.plot(epochs, val_accuracy, 'r', label='Validation accuracy')
ax1.set_title('Training and validation accuracy')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(epochs, loss, 'b', label='Training loss')
ax2.plot(epochs, val_loss, 'r', label='Validation loss')
ax2.set_title('Training and validation loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.legend()

fig.suptitle('Training and validation metrics', fontsize=16)
plt.show()

## Test Data Predictions

In [ ]:
# Predict labelling with dataset of test images
class_names = sorted(train_df['Label'].unique())

# Convert indices to class names
label_to_index = {label: idx for idx, label in enumerate(class_names)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

pred = model.predict(test_ds, verbose=0)
pred_idx = np.argmax(pred, axis=1)

pred_labels = [index_to_label[i] for i in pred_idx]
print('Predicted Labels:', pred_labels[:5])

true_labels = []
for _, y in test_ds:
    true_labels.extend(y.numpy().tolist())

true_labels = [index_to_label[i] for i in true_labels]
print('     True Labels:', true_labels[:5])


In [ ]:
# Display 15 random pictures 
pred_probs = model.predict(test_ds, verbose=0)
pred_idx = np.argmax(pred_probs, axis=1)


for ax, idx in zip(axes.flat, random_index):
    ax.imshow(plt.imread(test_df.Filepath.iloc[idx]))
    true_label = test_df.Label.iloc[idx]
    pred_label = pred[idx]
    if true_label == pred_label:
        color = "green"
    else:
        color = "red"
    ax.set_title(f"True: {true_label}\nPredicted: {pred_label}", color=color)


columns = 5
rows = 3
fig, axes = plt.subplots(rows, columns, figsize=(15, 10),
                        subplot_kw={'xticks': [], 'yticks': []})

plt.show()
plt.tight_layout()